## Install Packages

In [ ]:
!pip install pubchempy
!pip install rdkit

## Imports

In [ ]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Draw, AllChem
from rdkit.Chem.Draw import IPythonConsole
from rdkit.Chem.MolStandardize import rdMolStandardize

# From A Messy Supplier File To A Modelling-Ready Table

A colleague has sent you a [CSV](https://en.wikipedia.org/wiki/Comma-separated_values) file of ten compounds pulled out of a bioactivity database, with structures as [SMILES](https://en.wikipedia.org/wiki/Simplified_Molecular_Input_Line_Entry_System) text, a few physicochemical columns and a measured potency.
It looks tidy in a spreadsheet, and it is not: one SMILES is not a molecule at all, two records are salts, one compound appears twice under two different spellings, and two columns are full of gaps.
Today you take that file apart with [pandas](https://pandas.pydata.org/docs/) and [RDKit](https://www.rdkit.org/docs/), decide what each column is for, repair the chemistry, remove the duplicate structure, and deal with the missing values honestly.

## Part 1 - What Is Actually In The File

The file below is what arrived.

Each row is one compound: a database identifier (`chembl_id`), a name, a SMILES string, two computed physicochemical properties, a measured aqueous solubility, and a potency reported as [pIC50](https://en.wikipedia.org/wiki/IC50) - the negative logarithm of the concentration that halves the activity of the target, so a larger number means a more potent compound.

| chembl_id | name | smiles | mw | logp | solubility_mg_ml | pIC50 |
|---|---|---|---|---|---|---|
| CHEMBL25 | aspirin | `CC(=O)Oc1ccccc1C(=O)O` | 180.16 | 1.19 | 3.0 | 5.2 |
| CHEMBL113 | caffeine | `Cn1cnc2c1c(=O)n(C)c(=O)n2C` | 194.19 | -0.07 | | 4.8 |
| CHEMBL521 | ibuprofen | `CC(C)Cc1ccc(cc1)C(C)C(=O)O` | 206.28 | | | 4.5 |
| CHEMBL1004 | diphenhydramine HCl | `CN(C)CCOC(c1ccccc1)c1ccccc1.Cl` | 291.82 | | | 7.1 |
| CHEMBL2001 | aspirin (repeat) | `OC(=O)c1ccccc1OC(C)=O` | 180.16 | 1.19 | 2.8 | 5.6 |
| CHEMBL714 | salbutamol sulfate | `CC(C)(C)NCC(O)c1ccc(O)c(CO)c1.OS(=O)(=O)O` | 576.70 | | | 7.4 |
| CHEMBL9999 | junk entry | `C1CC1CC(((` | | | | 6.0 |
| CHEMBL112 | paracetamol | `CC(=O)Nc1ccc(O)cc1` | 151.16 | 0.46 | 14.0 | 5.0 |
| CHEMBL154 | naproxen | `COc1ccc2cc(ccc2c1)C(C)C(=O)O` | 230.26 | 3.18 | | 5.9 |
| CHEMBL118 | celecoxib | `Cc1ccc(cc1)c1cc(nn1c1ccc(cc1)S(N)(=O)=O)C(F)(F)F` | 381.37 | 3.90 | | 7.8 |

*WARNING:* An empty cell in a CSV file becomes a missing value once [pandas reads it](https://pandas.pydata.org/pandas-docs/version/2.3/reference/api/pandas.read_csv.html), and a nonsense SMILES becomes an ordinary string that pandas is perfectly happy with.

Part of this warning is, that we must *always* make sure to clean our data, before we even work with it.


### The File
The contents from the table above is stored in a file `library.csv` by the following code:

In [ ]:
csv_text = """chembl_id,name,smiles,mw,logp,solubility_mg_ml,pIC50
CHEMBL25,aspirin,CC(=O)Oc1ccccc1C(=O)O,180.16,1.19,3.0,5.2
CHEMBL113,caffeine,Cn1cnc2c1c(=O)n(C)c(=O)n2C,194.19,-0.07,,4.8
CHEMBL521,ibuprofen,CC(C)Cc1ccc(cc1)C(C)C(=O)O,206.28,,,4.5
CHEMBL1004,diphenhydramine HCl,CN(C)CCOC(c1ccccc1)c1ccccc1.Cl,291.82,,,7.1
CHEMBL2001,aspirin (repeat),OC(=O)c1ccccc1OC(C)=O,180.16,1.19,2.8,5.6
CHEMBL714,salbutamol sulfate,CC(C)(C)NCC(O)c1ccc(O)c(CO)c1.OS(=O)(=O)O,576.70,,,7.4
CHEMBL9999,junk entry,C1CC1CC(((,,,,6.0
CHEMBL112,paracetamol,CC(=O)Nc1ccc(O)cc1,151.16,0.46,14.0,5.0
CHEMBL154,naproxen,COc1ccc2cc(ccc2c1)C(C)C(=O)O,230.26,3.18,,5.9
CHEMBL118,celecoxib,Cc1ccc(cc1)c1cc(nn1c1ccc(cc1)S(N)(=O)=O)C(F)(F)F,381.37,3.90,,7.8
"""
with open("library.csv", "w") as fh:
    fh.write(csv_text)

#### Questions
1. Complete the line below so `library.csv` is read into a DataFrame called `records`.
2. We print the `records` but use the `records.head()` instead of just `record`. What is the effect? and is data missing? *hint:* you can [read the documentation](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.head.html) to get an idea.


In [ ]:
records = pd.read_csv(_)
print(records._())

### Identifiers, Features, and Endpoints
Every column in a dataset like this plays one of three roles.
An **identifier** names the row and is never used as an input to a model.
A **feature** is a property you would feed into a calculation or a model.
An **endpoint** is the measured biological result you want to explain or predict.

From a Data Science perspective, these are three core ideas you need to be able to work with.

#### Questions

3. Name the **identifier** columns, **feature** columns and **endpoint** columns in the data we just loaded in.
4. `solubility_mg_ml` was measured in a laboratory, while `mw` and `logp` were computed from the structure. Does that difference change which of the three roles they play?


In [ ]:
print("identifiers:", ["chembl_id", ...])
print("features:", [...])
print("endpoint:", ...)

### The SMILES Column
The `smiles` column looks like chemistry, but at this point it is text and nothing else.
Python can tell you how many characters are in `CC(=O)Oc1ccccc1C(=O)O`, but it cannot tell you how many carbon atoms are in aspirin until [`Chem.MolFromSmiles`](https://www.rdkit.org/docs/source/rdkit.Chem.rdmolfiles.html) has read the string and built the atoms and bonds.

This parsing step is crucial because it is also the only validity check the file will ever get in terms of valid chemistry.

#### Questions

6. Aspirin is $\mathrm{C_9H_8O_4}$, so it has 13 heavy atoms. Predict what `len(smiles_text)` prints, and why it is not 13.
7. Complete the two marked lines so the block parses that same string and prints its heavy-atom count.
8. The last line is just the variable name with no `print`. What appears?


In [ ]:
smiles_text = records["smiles"].iloc[0]
print("characters in the string:", len(smiles_text))
mol = Chem.MolFromSmiles(_)
print("heavy atoms after parsing:", mol._())
mol

## Part 2 - Cleaning the Data Set
---

For this part, having worked through videos [1.6](https://youtu.be/0MIv6ibqKso) and [2.4](https://youtu.be/NSQv0xdhhHs) at least will make this a lot easier.

---

Cleaning a chemical table happens in a fixed order:
  1. First parse every SMILES and drop what fails, because every later step needs a molecule object.
  2. Then strip salts down to the active organic species.
  3. Only then canonicalise and look for duplicates.

Step 1 is dangerous precisely because it is quiet: `Chem.MolFromSmiles` returns `None` for a string it cannot read instead of raising an error, so a broken record survives in your table until something else breaks much later.

#### Questions

1. Add a `mol` column to `records` by parsing every SMILES string through the `Chem.MolFromSmiles` with a list comprehension.
2. Write the loop that walks `name`, `smiles` and `mol` together with `zip` and prints a warning for every record whose molecule is `None`.
3. Comment on the result.

In [ ]:
records["_"] = [Chem.MolFromSmiles(smiles) for smiles in records["smiles"]]

# now we loop through each row of the three important columns and only print if something went wrong.
for name, smiles, mol in zip(records["_"], records["_"], records["mol"]):
    if mol is _:
        print("invalid SMILES for", name, ":", smiles)

#### Questions
We will now make a clean dataframe without bad molecules and call it `clean`.

  4. Predict how many rows `clean` will have based on the calculations and data available to you *above* here. Do not run the code just yet.

One way we filter code is to mark rows as *something we want to include* or *something we want to exclude*.
We have previously done this with code like `records["mw"] > 250.0` for example.
Let's dive a little into what actually happens.

  5. Insert the code above and comment on the result.


Using the above as a filter was typically done using the code presented below:

In [ ]:
mass_filter = records["mw"] > 250
records[_]

#### Questions
  6. Explain what happend to your data

### Ensuring That the Bogus Molecule does Not Survive
Let us make our own filter but for whether a molecule was valid or not.

#### Questions
  7. Construct a list `[]` named `clean_filter` with either `True` or `False` depending on whether a given row has a molecule that valid or not. *hint:* A valid molecule would mean that `True` is put into the list.

In [ ]:
clean_filter = []
for molecule in records["mol"]:
    if molecule is _:
        clean_filter.append(False)
    _:
        clean_filter.append(True)
clean_filter

#### Questions
  8. Apply the new `clean_filter` to records and store in a new `clean_records` dataframe. Comment on the results.

In [ ]:
clean_records = records[_].copy()
clean_records

### Displaying the Good Molecules
Use [the solutions from the old exercise](https://colab.research.google.com/drive/1UesS14AkJJghhXw1tDb1Bpj2Z0mfCk-r#scrollTo=b569c00c) to generate a grid of all the good molecules.

#### Questions
  9. Use the column `mol` in `clean_records` from above, go generate a grid of molecules. Similarly to last time, the molecule names should be displayed under each.

In [ ]:
mols = list(clean_records["mol"].values)
names = list(clean_records["name"].values)
grid = Draw.MolsToGridImage(mols, molsPerRow=3, subImgSize=(220, 180), legends=names)  #! tile the ten molecules with their names
grid

#### Questions
  10. Are there molecules that trouble you?

### Dealing with Salts
A central cleaning task for your data is to ensure that the data is actually representing the molecules you want.
In the figure above, you clearly observe that two molecules are salts.

In a SMILES string, a dot (`.`) separates [disconnected components](https://www.daylight.com/dayhtml/doc/theory/theory.smiles.html#RTFToC20), so `CN(C)CCOC(c1ccccc1)c1ccccc1.Cl` is the [diphenhydramine](https://en.wikipedia.org/wiki/Diphenhydramine) cation *and* a chloride atom, stored as one record. The counter-ion contributes to molecular weight, to logP, to every descriptor you *compute* and it is not the species that binds the target, so for modelling we keep the *largest part of the molecule* and discard the rest.

The function `rdMolStandardize.LargestFragmentChooser` from RDKit does this ([video 2.4](https://youtu.be/NSQv0xdhhHs)).
You build one chooser object and call `choose(mol)` on each molecule, getting the biggest connected fragment back.

#### Questions
Diphenhydramine hydrochloride is stored with the chloride attached as entry number 4 in the `clean_records` dataframe.

  11. How many atoms should there be less after we have selected only the largest fragment?
  12. Fix the code below.


In [ ]:
chooser = rdMolStandardize.LargestFragmentChooser()
salt_mol = clean_records["mol"].iloc[3]  #! 4th item is number 3 - remember we count from 0
parent_mol = chooser.choose(_)  #! keep the biggest connected fragment

print("with salt:", _.GetNumAtoms(), "heavy atoms")
print("parent:", _.GetNumAtoms(), "heavy atoms")

### Canonical SMILES of the parent fragment
So far, we have used `Chem.MolFromSmiles` to convert a SMILES string, into a molecule.
There is a function that we will use now `Chem.MolToSmiles` that turns a molecule back into a SMILES string.
Calling this funcion *always* produces the **canonical** form of a SMILES string, .i.e., RDKit's agreed spelling for a SMILES string.

Our task now is to run every parent fragment through this step of canonicalizing the molecules to give a new column (`canonical_smiles`) in which two records of the same compound are guaranteed to look identical.

### Questions

  13. Write the loop that takes each molecule in `clean["mol"]`, chooses its largest fragment, and appends the canonical SMILES of that fragment to `parent_smiles`.
  14. Store the finished list as a new column `canonical_smiles` and print `name`, `smiles` and `canonical_smiles` side by side.


In [ ]:
canon_smiles = []
for molecule in clean_records["mol"]:
    parent_molecule = chooser.choose(_)  #! strip the salt off each molecule
    canon_smiles.append(Chem.MolToSmiles(parent_molecule))  #!  and store its canonical SMILES

clean_records["canonical_smiles"] = canon_smiles  #! attach the list as a column

Let us now compare the SMILES strings with each other.
#### Questions
  15. Create a list of column names to keep by extending the list below.
  16. Comment on the rows with aspirin, for example.

In [ ]:
columns_to_show = ["name", "_", "..."]
clean_records[columns_to_show]